## 1. Imports



In [52]:
import re
import numpy as np
import tensorflow as tf
import pandas as pd

from tensorflow.keras.layers import (
    Input,
    Embedding,
    Dense,
    Dropout,
    LayerNormalization,
    MultiHeadAttention,
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


In [53]:
# ============================================================
# 1. LOAD DATASET
# ============================================================

dataset = pd.read_csv("en_ta_small_clean_test_dataset.csv")

english_sentences = dataset["english"].astype(str).tolist()
tamil_sentences = dataset["tamil"].astype(str).tolist()

assert len(english_sentences) == len(tamil_sentences)

print("Training pairs:", len(english_sentences))

for i in range(min(5, len(english_sentences))):
    print("English:", english_sentences[i])
    print("Tamil:", tamil_sentences[i])
    print()

#Al so use this 
# english_sentences = [
#     "I love this phone.",
#     "The phone has a good camera.",
#     "The project will start tomorrow.",
#     "I need to finish this work today.",
#     "Can you send the file tomorrow?",
#     "We need more time to complete the project.",
#     "The time frame for implementation is one year.",
#     "This is a good phone.",
#     "The camera is very good.",
#     "I need the file today.",
#     "The project is ready.",
#     "We will start the work tomorrow.",
#     "English"
# ]

# tamil_sentences = [
#     "எனக்கு இந்த தொலைபேசி மிகவும் பிடிக்கும்.",
#     "இந்த தொலைபேசியில் நல்ல கேமரா உள்ளது.",
#     "திட்டம் நாளை தொடங்கும்.",
#     "நான் இந்த வேலையை இன்று முடிக்க வேண்டும்.",
#     "நீங்கள் கோப்பை நாளை அனுப்ப முடியுமா?",
#     "திட்டத்தை முடிக்க எங்களுக்கு அதிக நேரம் தேவை.",
#     "செயல்படுத்துவதற்கான காலக்கெடு ஒரு வருடம்.",
#     "இது ஒரு நல்ல தொலைபேசி.",
#     "கேமரா மிகவும் நன்றாக உள்ளது.",
#     "எனக்கு இன்று கோப்பு தேவை.",
#     "திட்டம் தயாராக உள்ளது.",
#     "நாங்கள் நாளை வேலையை தொடங்குவோம்.",
#     "ஆங்கிலம்"
# ]

# assert len(english_sentences) == len(tamil_sentences)
# print("Training pairs:", len(english_sentences))

Training pairs: 101
English: How are you?
Tamil: நீங்கள் எப்படி இருக்கிறீர்கள்?

English: I am fine.
Tamil: நான் நலமாக இருக்கிறேன்.

English: What is your name?
Tamil: உங்கள் பெயர் என்ன?

English: My name is Ravi.
Tamil: என் பெயர் ரவி.

English: Where are you going?
Tamil: நீங்கள் எங்கே செல்கிறீர்கள்?



In [54]:
# ============================================================
# 2. ADD SOS / EOS TO TAMIL TARGETS
# ============================================================

tamil_sent_SE = [
    f"<sos> {text} <eos>"
    for text in tamil_sentences
]

print("English:", english_sentences[0])
print("Tamil:", tamil_sent_SE[0])


English: How are you?
Tamil: <sos> நீங்கள் எப்படி இருக்கிறீர்கள்? <eos>


In [55]:
# ============================================================
# 3. TOKENIZERS
# ============================================================

english_tokenizer = Tokenizer(
    filters="",
    oov_token="<unk>",
)
tamil_tokenizer = Tokenizer(
    filters="",
    oov_token="<unk>",
)

english_tokenizer.fit_on_texts(english_sentences)
tamil_tokenizer.fit_on_texts(tamil_sent_SE)

english_sequences = english_tokenizer.texts_to_sequences(english_sentences)
tamil_sequences = tamil_tokenizer.texts_to_sequences(tamil_sent_SE)

english_vocab_size = min(
    len(english_tokenizer.word_index) + 1,
    MAX_ENGLISH_VOCAB,
)
tamil_vocab_size = min(
    len(tamil_tokenizer.word_index) + 1,
    MAX_TAMIL_VOCAB,
)

print("English vocabulary used by model:", english_vocab_size)
print("Tamil vocabulary used by model:", tamil_vocab_size)


English vocabulary used by model: 172
Tamil vocabulary used by model: 179


In [74]:
# ============================================================
# 4. MAXIMUM SEQUENCE LENGTHS
# ============================================================

max_input_seq_length = max(
    len(x) for x in english_sequences
)

max_output_seq_length = max(
    len(x) for x in tamil_sequences
)

print("Max English length:", max_input_seq_length)
print("Max Tamil length:", max_output_seq_length)


Max English length: 8
Max Tamil length: 7


In [56]:
# ============================================================
# 5. PADDING
# ============================================================

input_sequences = pad_sequences(
    english_sequences,
    maxlen=max_input_seq_length,
    padding="post",
    truncating="post",
)

output_sequences = pad_sequences(
    tamil_sequences,
    maxlen=max_output_seq_length,
    padding="post",
    truncating="post",
)

print("Input shape:", input_sequences.shape)
print("Output shape:", output_sequences.shape)


Input shape: (101, 32)
Output shape: (101, 32)


In [57]:
# ============================================================
# 6. DECODER INPUT / TARGET
# ============================================================

decoder_input_sequences = output_sequences[:, :-1]
decoder_target_sequences = output_sequences[:, 1:]

print("Encoder input:", input_sequences.shape)
print("Decoder input:", decoder_input_sequences.shape)
print("Decoder target:", decoder_target_sequences.shape)

sample_weight = (decoder_target_sequences != 0).astype("float32")


Encoder input: (101, 32)
Decoder input: (101, 31)
Decoder target: (101, 31)


In [58]:
# ============================================================
# 7. POSITIONAL EMBEDDING
# ============================================================

class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, vocab_size, sequence_length, d_model, **kwargs):
        super().__init__(**kwargs)
        self.token_embedding = Embedding(vocab_size, d_model, mask_zero=True)
        self.position_embedding = Embedding(sequence_length, d_model)
        self.sequence_length = sequence_length

    def call(self, inputs):
        length = tf.shape(inputs)[-1]
        positions = tf.range(start=0, limit=length, delta=1)
        positions = self.position_embedding(positions)
        tokens = self.token_embedding(inputs)
        return tokens + positions


In [59]:
# ============================================================
# 8. TRANSFORMER ENCODER BLOCK
# ============================================================

# Each encoder block contains:
# 1. Multi-Head Self-Attention
# 2. Residual connection + LayerNorm
# 3. Feed-Forward Network
# 4. Residual connection + LayerNorm

class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout,
        )
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(d_model),
        ])
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout)
        self.dropout2 = Dropout(dropout)

    def call(self, x, training=False, attention_mask=None):
        attn_output = self.attention(
            query=x,
            key=x,
            value=x,
            attention_mask=attention_mask,
            training=training,
        )
        x = self.norm1(x + self.dropout1(attn_output, training=training))

        ffn_output = self.ffn(x, training=training)
        x = self.norm2(x + self.dropout2(ffn_output, training=training))
        return x


In [60]:
# ============================================================
# 9.TRANSFORMER DECODER BLOCK
# ============================================================

# The decoder has three important operations:
# 1. Causal self-attention — prevents looking at future target tokens.
# 2. Cross-attention — attends to the encoder output.
# 3. Feed-forward network.
    
class TransformerDecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)

        self.self_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout,
        )

        self.cross_attention = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=d_model // num_heads,
            dropout=dropout,
        )

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(d_model),
        ])

        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.norm3 = LayerNormalization(epsilon=1e-6)

        self.dropout1 = Dropout(dropout)
        self.dropout2 = Dropout(dropout)
        self.dropout3 = Dropout(dropout)

    def call(
        self,
        x,
        encoder_output,
        training=False,
        self_attention_mask=None,
        cross_attention_mask=None,
    ):
        attn1 = self.self_attention(
            query=x,
            key=x,
            value=x,
            attention_mask=self_attention_mask,
            training=training,
        )
        x = self.norm1(x + self.dropout1(attn1, training=training))

        attn2 = self.cross_attention(
            query=x,
            key=encoder_output,
            value=encoder_output,
            attention_mask=cross_attention_mask,
            training=training,
        )
        x = self.norm2(x + self.dropout2(attn2, training=training))

        ffn_output = self.ffn(x, training=training)
        x = self.norm3(x + self.dropout3(ffn_output, training=training))

        return x


## 10. Causal mask

At decoder position `t`, the Transformer must not see target tokens after `t`.


In [61]:
# ============================================================
# 10. CAUSAL MASK 
# ============================================================

#At decoder position t, the Transformer must not see target tokens after t

def causal_mask(length):
    # 1 = allowed, 0 = blocked
    i = tf.range(length)[:, None]
    j = tf.range(length)[None, :]
    return tf.cast(i >= j, tf.int32)


In [62]:
# ============================================================
# 10. TRANSFORMER
# ============================================================

# Encoder: English IDs → embedding + position → encoder block
# Decoder: Tamil decoder IDs → embedding + position → masked self-attention → cross-attention → output logits

class Transformer(tf.keras.Model):
    def __init__(
        self,
        english_vocab_size,
        tamil_vocab_size,
        max_encoder_length,
        max_decoder_length,
        d_model=64,
        num_heads=4,
        ff_dim=128,
        dropout=0.1,
        **kwargs,
    ):
        super().__init__(**kwargs)

        self.encoder_embedding = PositionalEmbedding(
            english_vocab_size,
            max_encoder_length,
            d_model,
        )

        self.decoder_embedding = PositionalEmbedding(
            tamil_vocab_size,
            max_decoder_length,
            d_model,
        )

        self.encoder = TransformerEncoderBlock(
            d_model, num_heads, ff_dim, dropout
        )

        self.decoder = TransformerDecoderBlock(
            d_model, num_heads, ff_dim, dropout
        )

        self.output_layer = Dense(tamil_vocab_size)

    def call(self, inputs, training=False):
        encoder_tokens, decoder_tokens = inputs

        encoder_x = self.encoder_embedding(encoder_tokens)
        decoder_x = self.decoder_embedding(decoder_tokens)

        # Padding masks.
        encoder_padding = tf.cast(
            encoder_tokens != 0, tf.int32
        )[:, None, :]
        decoder_padding = tf.cast(
            decoder_tokens != 0, tf.int32
        )[:, None, :]

        # Encoder self-attention mask: query positions × key positions.
        encoder_mask = tf.cast(
            encoder_padding[:, :, None] * encoder_padding[:, None, :],
            tf.int32,
        )

        # Decoder causal mask.
        target_length = tf.shape(decoder_tokens)[1]
        cmask = causal_mask(target_length)[None, :, :]

        # Decoder padding mask combined with causal mask.
        decoder_mask = cmask * tf.cast(
            decoder_padding[:, None, :],
            tf.int32,
        )

        # Cross-attention: decoder queries may attend to non-padding encoder keys.
        cross_mask = tf.cast(
            encoder_padding[:, None, :],
            tf.int32,
        )

        encoder_output = self.encoder(
            encoder_x,
            training=training,
            attention_mask=encoder_mask,
        )

        decoder_output = self.decoder(
            decoder_x,
            encoder_output,
            training=training,
            self_attention_mask=decoder_mask,
            cross_attention_mask=cross_mask,
        )

        return self.output_layer(decoder_output)


In [63]:
# ============================================================
# 11. BUILD THE SMALL TRANSFORMER
# ============================================================

D_MODEL = 64
NUM_HEADS = 4
FF_DIM = 128
DROPOUT = 0.1

transformer = Transformer(
    english_vocab_size=english_vocab_size,
    tamil_vocab_size=tamil_vocab_size,
    max_encoder_length=max_input_seq_length,
    max_decoder_length=max_output_seq_length - 1,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    dropout=DROPOUT,
)

# Build once.
dummy_encoder = tf.constant(input_sequences[:1], dtype=tf.int32)
dummy_decoder = tf.constant(decoder_input_sequences[:1], dtype=tf.int32)
dummy_logits = transformer([dummy_encoder, dummy_decoder])

transformer.summary()
print("Logits shape:", dummy_logits.shape)


Model: "transformer_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ positional_embedding_4          │ ?                      │        13,056 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ positional_embedding_5          │ ?                      │        13,440 │
│ (PositionalEmbedding)           │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_2     │ ?                      │        33,472 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_decoder_block_2     │ ?                      │        50,240 │
│ (TransformerDecoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (1, 31, 179)           │        11,635 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121,843 (475.95 KB)

 Trainable params: 121,843 (475.95 KB)

 Non-trainable params: 0 (0.00 B)

Logits shape: (1, 31, 179)


In [64]:
# ============================================================
# 12. COMPILE
# ============================================================

# Targets are integer token IDs, so use sparse categorical crossentropy.
# The model returns logits, not softmax probabilities.

loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True
)

transformer.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=loss_fn,
    weighted_metrics=["accuracy"],
)


In [69]:
# ============================================================
# 13. TRAIN
# ============================================================


EPOCHS = 20
BATCH_SIZE = 5

history = transformer.fit(
    [input_sequences, decoder_input_sequences],
    decoder_target_sequences,
    sample_weight=sample_weight,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True,
    verbose=1,
)


Epoch 1/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9927 - loss: 0.0175
Epoch 2/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9951 - loss: 0.0164
Epoch 3/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9951 - loss: 0.0148
Epoch 4/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9951 - loss: 0.0120
Epoch 5/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0102
Epoch 6/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9976 - loss: 0.0102
Epoch 7/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9951 - loss: 0.0098
Epoch 8/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0078
Epoch 9/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9976 - loss: 0.0100
Epoch 10/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9976 - loss: 0.0077
Epoch 11/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0076
Epoch 12/20
21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - lo

In [70]:
# ============================================================
# 14. INFERENCE MODELS
# ============================================================

#During inference, the decoder starts with <sos> and predicts one token at a time until <eos>.

sos_id = tamil_tokenizer.word_index["<sos>"]
eos_id = tamil_tokenizer.word_index["<eos>"]

def translate(sentence, max_length=31):
    encoder_ids = english_tokenizer.texts_to_sequences([sentence])
    encoder_ids = pad_sequences(
        encoder_ids,
        maxlen=max_input_seq_length,
        padding="post",
        truncating="post",
    )

    decoder_ids = [sos_id]

    for _ in range(max_length):
        decoder_input = pad_sequences(
            [decoder_ids],
            maxlen=max_output_seq_length - 1,
            padding="post",
            truncating="post",
        )

        logits = transformer(
            [
                tf.constant(encoder_ids, dtype=tf.int32),
                tf.constant(decoder_input, dtype=tf.int32),
            ],
            training=False,
        )

        step = len(decoder_ids) - 1
        next_id = int(tf.argmax(logits[0, step]).numpy())

        if next_id == eos_id or next_id == 0:
            break

        decoder_ids.append(next_id)

    words = []
    index_to_word = {
        idx: word
        for word, idx in tamil_tokenizer.word_index.items()
        if idx < tamil_vocab_size
    }

    for idx in decoder_ids[1:]:
        if idx == eos_id:
            break
        words.append(index_to_word.get(idx, "<unk>"))

    return " ".join(words)


In [71]:
# ============================================================
# 15. TEST TRANSLATION
#============================================================

test_sentences = [ 
    "I am going home.",
    "This phone is good.",
    "The phone has a good camera.",
    "The time frame for implementation is one year.,"
]

for sentence in test_sentences:
    print("English:", sentence)
    print("Tamil:", translate(sentence))
    print()


English: I am going home.
Tamil: நான் வீட்டிற்குச் செல்கிறேன்.

English: This phone is good.
Tamil: இந்த தொலைபேசி நன்றாக இருக்கிறது.

English: The phone has a good camera.
Tamil: இந்த தொலைபேசியில் நல்ல கேமரா உள்ளது.

English: The time frame for implementation is one year.,
Tamil: செயல்படுத்துவதற்கான காலக்கெடு ஒரு வருடம்.



In [75]:
# ============================================================
# 16. TEST NEW WORDS
#============================================================

test_sentences = [
    "This dataset contains several English sentences",
    "English",
]

for sentence in test_sentences:
    print("English:", sentence)
    print("Tamil:", translate(sentence))
    print()


English: This dataset contains several English sentences
Tamil: இந்த தரவுத்தொகுப்பு சுத்தமாக உள்ளது.

English: English
Tamil: நான் நலமாக இருக்கிறேன்.



## 17. Important learning checkpoints

Before moving to the large dataset, understand these five Transformer concepts:

1. **Token embedding** — converts token IDs into vectors.
2. **Positional information** — tells the Transformer where each token occurs.
3. **Self-attention** — tokens attend to other tokens in the same sequence.
4. **Cross-attention** — the Tamil decoder attends to the English encoder output.
5. **Causal masking** — the decoder cannot see future Tamil target tokens during training.

Only after this small model is working should we scale vocabulary, sequence length, layers, model dimension, batch size, and dataset size.
